# Phase 1: Machine Learning Model Training — Baseline & Tree Models


# Import Libraries


In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


# Load Prepared Data


In [2]:
ROOT_DIR = None
for candidate in ['.', '..', '../..']:
    if os.path.exists(os.path.join(candidate, 'data', 'GeneratedOutput', '02.Preprocessing&FeatureSelection')):
        ROOT_DIR = os.path.abspath(candidate)
        break
if ROOT_DIR is None:
    ROOT_DIR = os.path.abspath('.')

DATA_DIR = os.path.join(ROOT_DIR, 'data', 'GeneratedOutput')
train_path = os.path.join(DATA_DIR, '02.Preprocessing&FeatureSelection', 'train.parquet')
val_path = os.path.join(DATA_DIR, '02.Preprocessing&FeatureSelection', 'val.parquet')

train_data = pd.read_parquet(train_path)
val_data = pd.read_parquet(val_path)

print(f"Training data shape  : {train_data.shape[0]:,} rows x {train_data.shape[1]} columns")
print(f"Validation data shape: {val_data.shape[0]:,} rows x {val_data.shape[1]} columns")

Training data shape  : 1,972,250 rows x 22 columns
Validation data shape: 422,625 rows x 22 columns


# Select Features and Target


In [3]:
features = [
    "ph",
    "temperature_c",
    "do_mg_l",
    "bod_mg_l",
    "ammonia_mg_l",
    "orthophosphate_mg_l",
    "nitrogen_mg_l",
    "nitrate_mg_l",
    "np_ratio",
    "bod_do_ratio",
    "do_deficit",
    "month",
    "quarter",
    "month_sin",
    "month_cos",
]
target = "ccme_values"

X_train = train_data[features]
y_train = train_data[target]

X_val = val_data[features]
y_val = val_data[target]

print(f"Selected {len(features)} input features.")
print(f"Target column: {target}")


Selected 15 input features.
Target column: ccme_values


# Train a Mean Baseline Regressor


In [4]:
baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train, y_train)

print(f"Mean baseline fitted. Constant prediction value: {baseline_model.constant_[0][0]:.4f}")


Mean baseline fitted. Constant prediction value: 84.9993


# Train Random Forest Regressor


In [5]:
rf_model = RandomForestRegressor(
    n_estimators=30,
    max_depth=14,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)

print(f"Training Random Forest on full {len(X_train):,} rows...")
rf_model.fit(X_train, y_train)
print("Random Forest training complete.")

Training Random Forest on full 1,972,250 rows...


Random Forest training complete.


# Train XGBoost Regressor


In [6]:
xgb_model = XGBRegressor(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    early_stopping_rounds=20,
    random_state=42,
    n_jobs=-1,
)

print("Training XGBoost...")
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)
print(f"XGBoost training complete. Best iteration: {xgb_model.best_iteration}")

Training XGBoost...


[0]	validation_0-rmse:16.26859


[50]	validation_0-rmse:0.89729


[100]	validation_0-rmse:0.72492


[149]	validation_0-rmse:0.68833


XGBoost training complete. Best iteration: 149


# Save Trained Models and Validation Predictions


In [7]:
output_dir = os.path.join(DATA_DIR, "03.ModelTraining")
os.makedirs(output_dir, exist_ok=True)

rf_path = os.path.join(output_dir, "random_forest_model.joblib")
joblib.dump(rf_model, rf_path)

xgb_path = os.path.join(output_dir, "xgboost_model.json")
xgb_model.save_model(xgb_path)

val_predictions = pd.DataFrame({
    "actual": y_val.to_numpy(),
    "pred_baseline": baseline_model.predict(X_val),
    "pred_rf": rf_model.predict(X_val),
    "pred_xgb": xgb_model.predict(X_val),
})
preds_path = os.path.join(output_dir, "tree_val_predictions.parquet")
val_predictions.to_parquet(preds_path, index=False)

print(f"Saved artifacts to {output_dir}:")
print(f"  • {os.path.basename(rf_path)}")
print(f"  • {os.path.basename(xgb_path)}")
print(f"  • {os.path.basename(preds_path)} ({len(val_predictions):,} rows)")


Saved artifacts to /Users/amansoni/Documents/Development/Personal Projects/Semester_5_AML_DL/data/GeneratedOutput/03.ModelTraining:
  • random_forest_model.joblib
  • xgboost_model.json
  • tree_val_predictions.parquet (422,625 rows)


# Basic Training Check


In [8]:
loaded_rf = joblib.load(rf_path)
sample_rf_pred = loaded_rf.predict(X_val.iloc[:5])

loaded_xgb = XGBRegressor()
loaded_xgb.load_model(xgb_path)
sample_xgb_pred = loaded_xgb.predict(X_val.iloc[:5])

print("Sample actual values   :", np.round(y_val.iloc[:5].to_numpy(), 2))
print("Random Forest predicted:", np.round(sample_rf_pred, 2))
print("XGBoost predicted      :", np.round(sample_xgb_pred, 2))
print("\nAll models loaded successfully and generated valid predictions.")


Sample actual values   : [ 84.96 100.   100.   100.    72.45]
Random Forest predicted: [ 85.26 100.   100.   100.    72.18]
XGBoost predicted      : [ 85.04  98.85 100.    99.94  72.11]

All models loaded successfully and generated valid predictions.
